In [2]:
! pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib openpyxl

In [4]:
! pip install pandas

  Using cached pandas-3.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.4.6-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached pandas-3.0.3-cp314-cp314-macosx_11_0_arm64.whl (9.9 MB)
Using cached numpy-2.4.6-cp314-cp314-macosx_14_0_arm64.whl (5.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


In [3]:
import pandas as pd
import os
import json
import io
from googleapiclient.discovery import build
from google.oauth2 import service_account
from googleapiclient.http import MediaIoBaseDownload

# =============================================================
# GOOGLE DRIVE API SETUP
# =============================================================

# Define the scopes and the path to your service account credentials
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
SERVICE_ACCOUNT_FILE = 'my-project-41584-498506-8e338a591f9b.json' # <-- Place your Google Cloud credentials JSON here

def authenticate_drive():
    """Authenticates and returns the Google Drive service object."""
    creds = service_account.Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE, scopes=SCOPES)
    service = build('drive', 'v3', credentials=creds)
    return service

def download_file_from_drive(service, file_id, output_path):
    """Downloads a file from Google Drive by its ID."""
    print(f"Downloading file ID: {file_id} from Google Drive...")
    request = service.files().get_media(fileId=file_id)
    fh = io.FileIO(output_path, 'wb')
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while done is False:
        status, done = downloader.next_chunk()
        print(f"Download {int(status.progress() * 100)}%.")
    return output_path

# =============================================================
# ORIGINAL EXTRACTION LOGIC
# =============================================================

def extract_all_excel_data(file_path):
    """
    Extracts all data from every sheet in a single Excel workbook.
    """
    print(f"\n[{os.path.basename(file_path)}] Reading file...")
    
    try:
        all_sheets = pd.read_excel(file_path, sheet_name=None, engine='openpyxl')
        extracted_data = {}
        
        for sheet_name, df in all_sheets.items():
            # Drop entirely empty rows and columns
            df_cleaned = df.dropna(how='all', axis=0).dropna(how='all', axis=1)
            extracted_data[sheet_name] = df_cleaned
            print(f"  -> Tab '{sheet_name}': Found {len(df_cleaned)} rows.")
            
        return extracted_data

    except Exception as e:
        print(f"  -> An error occurred with this file: {e}")
        return None

# =============================================================
# EXECUTION
# =============================================================

if __name__ == "__main__":
    
    # 1. ADD YOUR GOOGLE DRIVE FILE IDs HERE
    # You can find the file ID in the shareable link of the Google Drive file
    # Example: https://drive.google.com/file/d/THIS_IS_THE_FILE_ID/view
    drive_files = {
        # "project_and_retainers_template.xlsx": "YOUR_FIRST_FILE_ID_HERE",
        # "Wohlig Active Employee Data.xlsx": "YOUR_SECOND_FILE_ID_HERE"
        "Wohlig Active Employee Data.xlsx": "1N--TipEgGAiMCeqji8vp49A_LGLCQT63"
    }
    
    output_json_file = "all_files_extracted_data.json"
    download_folder = "data"
    
    # Ensure download directory exists
    os.makedirs(download_folder, exist_ok=True)
    
    try:
        # Authenticate Drive API
        drive_service = authenticate_drive()
        master_json_data = {}
        
        # Loop through defined Drive files, download, and extract
        for file_name, file_id in drive_files.items():
            local_file_path = os.path.join(download_folder, file_name)
            
            # Download from Drive
            download_file_from_drive(drive_service, file_id, local_file_path)
            
            # Extract the data using the original function
            file_data = extract_all_excel_data(local_file_path)
            
            if file_data:
                json_ready_file_data = {}
                
                # Convert the dataframes to JSON-friendly dictionaries
                for sheet_name, df in file_data.items():
                    json_ready_file_data[sheet_name] = df.to_dict(orient='records')
                
                # Store it in the master dictionary under the file's name
                master_json_data[file_name] = json_ready_file_data
        
        # Save the massive master dictionary to a single JSON file
        print("\n--- Saving Master JSON ---")
        with open(output_json_file, 'w', encoding='utf-8') as json_file:
            json.dump(master_json_data, json_file, indent=4, default=str)
        print(f"Success! All files extracted and saved to: {output_json_file}")
        
    except Exception as e:
        print(f"Process failed: {e}")

Download 100%.

[Wohlig Active Employee Data.xlsx] Reading file...
  -> Tab 'Dashboard': Found 8 rows.
  -> Tab 'Active': Found 34 rows.
  -> Tab 'Exit': Found 0 rows.
  -> Tab 'Project': Found 2 rows.
  -> Tab 'Retainers': Found 3 rows.
  -> Tab 'Internal': Found 1 rows.

--- Saving Master JSON ---
Success! All files extracted and saved to: all_files_extracted_data.json
